<a href="https://colab.research.google.com/github/INDRAYUDH-SIKDAR/first-hackathon/blob/main/Helioscope.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio matplotlib

In [ ]:
import gradio as gr
import matplotlib.pyplot as plt

# --- 1. RESEARCH CONSTANTS ---
REGIONAL_DATA = {
    "Portland / Lake Oswego, OR (US)": {
        "latitude": 45.5,
        "irradiance": 4.0,
        "emission_factor": 0.288,
        "rate": 0.158,
        "currency": "$",
    },
    "Kolkata (Eastern India)": {
        "latitude": 22.7,
        "irradiance": 5.0,
        "emission_factor": 0.82,
        "rate": 7.2,
        "currency": "₹",
    }
}

EFFICIENCY = 0.18
LOSS_FRACTION = 0.15

# --- 2. BACKEND MATH ENGINE ---
def calculate_solar_impact(region, roof_area, consumption, cust_lat, cust_irr, cust_ef, cust_rate, cust_curr):
    # Route logic based on dropdown selection
    if region == "Custom Location":
        lat, H, ef = cust_lat, cust_irr, cust_ef
        rate, currency = cust_rate, cust_curr
    else:
        data = REGIONAL_DATA[region]
        lat, H, ef = data["latitude"], data["irradiance"], data["emission_factor"]
        rate, currency = data["rate"], data["currency"]

    # 1. Generation & Tilt
    optimal_tilt = round(lat * 0.9, 1)
    daily_gen = roof_area * H * EFFICIENCY * (1 - LOSS_FRACTION)
    monthly_gen = round(daily_gen * 30, 1)

    # 2. Consumption vs Surplus Routing
    usable_gen = min(monthly_gen, consumption)
    surplus_gen = max(0, monthly_gen - consumption)
    remaining_grid = max(0, consumption - usable_gen)

    # 3. Financials (Savings + Selling Surplus)
    monthly_savings = round(usable_gen * rate, 2)
    surplus_profit = round(surplus_gen * rate, 2)
    total_financial_value = monthly_savings + surplus_profit

    # 4. Carbon Impact
    co2_offset = round(monthly_gen * ef, 1)

    # 5. Format Summary output
    results = f"""
    ### 🌍 Regional Climate Profile: {region}
    * **Optimal Panel Tilt:** {optimal_tilt}°
    * **Estimated Monthly Generation:** {monthly_gen} kWh

    ### 💰 Financial Impact
    * **Grid Savings:** {currency}{monthly_savings:,.2f}
    * **Surplus Sold to Grid:** {currency}{surplus_profit:,.2f}
    * **Total Monthly Value:** **{currency}{total_financial_value:,.2f}**
    *(Based on local utility rate of {currency}{rate}/kWh)*

    ### 🌿 Carbon Displacement (The True Impact)
    * **CO₂ Offset per Month:** **{co2_offset:,.1f} kg of CO₂**
    *(Displacing local grid emissions at {ef} kg CO₂/kWh)*
    """

    # 6. Generate Matplotlib Chart
    fig, ax = plt.subplots(figsize=(6, 4), dpi=100)
    categories = ['Monthly\nDemand', 'Solar\nGenerated', 'Grid\nDependency', 'Surplus\nSold']
    values = [consumption, monthly_gen, remaining_grid, surplus_gen]
    colors = ['#64748b', '#f59e0b', '#ef4444', '#10b981']

    bars = ax.bar(categories, values, color=colors, width=0.5)
    ax.set_ylabel('Energy (kWh)', fontsize=11, fontweight='bold')
    ax.set_title('Grid Power vs. Solar Distribution', fontsize=12, fontweight='bold', pad=15)

    # Add values on top of bars
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2.0, yval + (max(values)*0.02 if max(values) > 0 else 0.5),
                f"{yval:.1f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.close(fig) # Prevent duplicate plotting in Colab

    return results, fig

# --- 3. FRONTEND UI WIRING ---
with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald")) as demo:
    gr.Markdown("# ☀️ HelioScope: Regional Solar & Carbon Forecaster")
    gr.Markdown("Select a region to see how the exact same roof yields vastly different carbon and financial impacts based on local grid realities.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Main Parameters")
            region_dropdown = gr.Dropdown(
                choices=list(REGIONAL_DATA.keys()) + ["Custom Location"],
                value="Portland / Lake Oswego, OR (US)",
                label="Select Geographic Region"
            )
            roof_area_slider = gr.Slider(
                minimum=10, maximum=150, value=40, step=1,
                label="Usable Roof Area (m²)"
            )
            consumption_slider = gr.Slider(
                minimum=100, maximum=1500, value=900, step=50,
                label="Monthly Electricity Consumption (kWh)"
            )

            with gr.Accordion("🛠️ Custom Location Overrides", open=False):
                gr.Markdown("*Only used if 'Custom Location' is selected above.*")
                c_lat = gr.Number(value=37.7, label="Latitude (°)")
                c_irr = gr.Number(value=4.5, label="Irradiance (kWh/m²/day)")
                c_ef = gr.Number(value=0.5, label="Grid Emission Factor (kg CO₂/kWh)")
                c_rate = gr.Number(value=0.20, label="Energy Rate")
                c_curr = gr.Textbox(value="$", label="Currency Symbol")

            calculate_btn = gr.Button("Calculate Impact", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### 📊 Forecast Results")
            output_display = gr.Markdown("...")
            plot_output = gr.Plot(label="Solar Offset Chart")

    # Connect inputs to the math engine
    calculate_btn.click(
        fn=calculate_solar_impact,
        inputs=[region_dropdown, roof_area_slider, consumption_slider,
                c_lat, c_irr, c_ef, c_rate, c_curr],
        outputs=[output_display, plot_output]
    )

demo.launch(debug=True)